# 06 — Late Data + Event Time

Purpose: handle events that arrive late.

Process schema:

```text
EVENT TIME     = when the business event happened
INGESTION TIME = when Spark received the event

BATCH ARRIVAL ORDER
+----------+---------+--------+---------------------+---------------------+
| event_id | user_id | amount | event_time          | ingestion_time      |
+----------+---------+--------+---------------------+---------------------+
| e1       | u1      | 10.0   | 2026-01-01 10:00:00 | 2026-01-01 10:01:00 |
| e2       | u1      | 20.0   | 2026-01-01 10:05:00 | 2026-01-01 10:06:00 |
| e3       | u1      | 30.0   | 2026-01-01 09:55:00 | 2026-01-01 10:20:00 | late but acceptable
| e4       | u2      | 40.0   | 2026-01-01 08:00:00 | 2026-01-01 10:30:00 | too late
+----------+---------+--------+---------------------+---------------------+

Rule:
- accept events if ingestion_time - event_time <= 30 minutes
- quarantine events later than 30 minutes
```

This notebook uses batch DataFrames to demonstrate the pattern.

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("medallion-late-data-event-time")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_medallion_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Input events

The data has both timestamps:

```text
event_time      = business time
ingestion_time  = technical arrival time
```

In [2]:
event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("event_time_raw", StringType(), False),
    StructField("ingestion_time_raw", StringType(), False),
])

rows = [
    ("e1", "u1", 10.0, "2026-01-01 10:00:00", "2026-01-01 10:01:00"),
    ("e2", "u1", 20.0, "2026-01-01 10:05:00", "2026-01-01 10:06:00"),
    ("e3", "u1", 30.0, "2026-01-01 09:55:00", "2026-01-01 10:20:00"),
    ("e4", "u2", 40.0, "2026-01-01 08:00:00", "2026-01-01 10:30:00"),
    ("e5", "u2", 50.0, "2026-01-01 10:25:00", "2026-01-01 10:31:00"),
]

events = (
    spark.createDataFrame(rows, event_schema)
    .withColumn("event_time", F.to_timestamp("event_time_raw"))
    .withColumn("ingestion_time", F.to_timestamp("ingestion_time_raw"))
    .drop("event_time_raw", "ingestion_time_raw")
)

events.show(truncate=False)
events.printSchema()

+--------+-------+------+-------------------+-------------------+
|event_id|user_id|amount|event_time         |ingestion_time     |
+--------+-------+------+-------------------+-------------------+
|e1      |u1     |10.0  |2026-01-01 10:00:00|2026-01-01 10:01:00|
|e2      |u1     |20.0  |2026-01-01 10:05:00|2026-01-01 10:06:00|
|e3      |u1     |30.0  |2026-01-01 09:55:00|2026-01-01 10:20:00|
|e4      |u2     |40.0  |2026-01-01 08:00:00|2026-01-01 10:30:00|
|e5      |u2     |50.0  |2026-01-01 10:25:00|2026-01-01 10:31:00|
+--------+-------+------+-------------------+-------------------+

root
 |-- event_id: string (nullable = false)
 |-- user_id: string (nullable = false)
 |-- amount: double (nullable = false)
 |-- event_time: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)



## Step 2 — Calculate lateness

```text
lateness_minutes = ingestion_time - event_time
```

In [3]:
events_with_lateness = (
    events
    .withColumn(
        "lateness_minutes",
        (F.col("ingestion_time").cast("long") - F.col("event_time").cast("long")) / 60.0
    )
)

events_with_lateness.orderBy("ingestion_time").show(truncate=False)

+--------+-------+------+-------------------+-------------------+----------------+
|event_id|user_id|amount|event_time         |ingestion_time     |lateness_minutes|
+--------+-------+------+-------------------+-------------------+----------------+
|e1      |u1     |10.0  |2026-01-01 10:00:00|2026-01-01 10:01:00|1.0             |
|e2      |u1     |20.0  |2026-01-01 10:05:00|2026-01-01 10:06:00|1.0             |
|e3      |u1     |30.0  |2026-01-01 09:55:00|2026-01-01 10:20:00|25.0            |
|e4      |u2     |40.0  |2026-01-01 08:00:00|2026-01-01 10:30:00|150.0           |
|e5      |u2     |50.0  |2026-01-01 10:25:00|2026-01-01 10:31:00|6.0             |
+--------+-------+------+-------------------+-------------------+----------------+



## Step 3 — Split accepted vs too late

Business rule:

```text
accepted if lateness_minutes <= 30
too late if lateness_minutes > 30
```

In [4]:
max_allowed_lateness_minutes = 30.0

accepted_events = events_with_lateness.filter(
    F.col("lateness_minutes") <= F.lit(max_allowed_lateness_minutes)
)

late_quarantine = (
    events_with_lateness
    .filter(F.col("lateness_minutes") > F.lit(max_allowed_lateness_minutes))
    .withColumn("error_reason", F.lit("too_late"))
)

accepted_events.show(truncate=False)
late_quarantine.show(truncate=False)

+--------+-------+------+-------------------+-------------------+----------------+
|event_id|user_id|amount|event_time         |ingestion_time     |lateness_minutes|
+--------+-------+------+-------------------+-------------------+----------------+
|e1      |u1     |10.0  |2026-01-01 10:00:00|2026-01-01 10:01:00|1.0             |
|e2      |u1     |20.0  |2026-01-01 10:05:00|2026-01-01 10:06:00|1.0             |
|e3      |u1     |30.0  |2026-01-01 09:55:00|2026-01-01 10:20:00|25.0            |
|e5      |u2     |50.0  |2026-01-01 10:25:00|2026-01-01 10:31:00|6.0             |
+--------+-------+------+-------------------+-------------------+----------------+

+--------+-------+------+-------------------+-------------------+----------------+------------+
|event_id|user_id|amount|event_time         |ingestion_time     |lateness_minutes|error_reason|
+--------+-------+------+-------------------+-------------------+----------------+------------+
|e4      |u2     |40.0  |2026-01-01 08:00:00|20

## Step 4 — Aggregate by event time

Gold aggregation should use `event_time`, not `ingestion_time`.

```text
Correct reporting grain = date(event_time)
```

In [5]:
gold_daily = (
    accepted_events
    .withColumn("event_date", F.to_date("event_time"))
    .groupBy("user_id", "event_date")
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount").alias("total_amount"),
    )
    .orderBy("user_id", "event_date")
)

gold_daily.show(truncate=False)

+-------+----------+-----------+------------+
|user_id|event_date|event_count|total_amount|
+-------+----------+-----------+------------+
|u1     |2026-01-01|3          |60.0        |
|u2     |2026-01-01|1          |50.0        |
+-------+----------+-----------+------------+



## Step 5 — Compare with wrong aggregation

This shows why ingestion time is not the same as business time.

In [6]:
wrong_gold_by_ingestion_date = (
    accepted_events
    .withColumn("ingestion_date", F.to_date("ingestion_time"))
    .groupBy("user_id", "ingestion_date")
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount").alias("total_amount"),
    )
    .orderBy("user_id", "ingestion_date")
)

wrong_gold_by_ingestion_date.show(truncate=False)

+-------+--------------+-----------+------------+
|user_id|ingestion_date|event_count|total_amount|
+-------+--------------+-----------+------------+
|u1     |2026-01-01    |3          |60.0        |
|u2     |2026-01-01    |1          |50.0        |
+-------+--------------+-----------+------------+



## Step 6 — Control totals

```text
input rows = accepted rows + late quarantine rows
```

In [7]:
control_totals_schema = StructType([
    StructField("metric", StringType(), False),
    StructField("value", LongType(), False),
])

control_totals = spark.createDataFrame(
    [
        ("input_rows", events.count()),
        ("accepted_rows", accepted_events.count()),
        ("late_quarantine_rows", late_quarantine.count()),
    ],
    control_totals_schema
)

control_totals.show(truncate=False)

+--------------------+-----+
|metric              |value|
+--------------------+-----+
|input_rows          |5    |
|accepted_rows       |4    |
|late_quarantine_rows|1    |
+--------------------+-----+



## Streaming note

In real Spark Structured Streaming this concept is usually implemented with watermarking:

```python
df.withWatermark("event_time", "30 minutes")
```

This notebook keeps it in batch form so the logic is visible.

## Final result

```text
EVENTS
  |
  v
CALCULATE LATENESS
  |
  +--> ACCEPTED EVENTS       lateness <= threshold
  |
  +--> LATE QUARANTINE       lateness > threshold
  |
  v
GOLD AGGREGATION BY event_time
```

In [8]:
spark.stop()